In [ ]:
# How does wildfires intensity vary accross Africa and How do the events differ in physical size, thermal intensity, or duration? in the last 5 days?"

In [38]:
import requests
import pandas as pd
import time

#import 
# Range parameter: 5 = last 5 days of detections, before 1
url_api = "https://firms.modaps.eosdis.nasa.gov/api/area/csv/dc38faa964bb2d75d020b913a3579aa3/VIIRS_SNPP_NRT/world/5"

response = requests.get(url_api)
print(response.status_code)
print(response.text[:200]) 

fires_df = pd.read_csv(url_api)
fires_df.info()
fires_df.head()



200
latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
68.64421,57.96012,328.76,0.57,0.52,2026-05-03,54,N,VIIRS,n,2.0NRT,268.08,2.46
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134253 entries, 0 to 134252
Data columns (total 14 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   latitude    134253 non-null  float64
 1   longitude   134253 non-null  float64
 2   bright_ti4  134253 non-null  float64
 3   scan        134253 non-null  float64
 4   track       134253 non-null  float64
 5   acq_date    134253 non-null  object 
 6   acq_time    134253 non-null  int64  
 7   satellite   134253 non-null  object 
 8   instrument  134253 non-null  object 
 9   confidence  134253 non-null  object 
 10  version     134253 non-null  object 
 11  bright_ti5  134253 non-null  float64
 12  frp         134253 non-null  float64
 13  daynight    134253 non-null  object 
dtypes: floa

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,68.64421,57.96012,328.76,0.57,0.52,2026-05-03,54,N,VIIRS,n,2.0NRT,268.08,2.46,D
1,68.64610,57.95589,327.43,0.57,0.52,2026-05-03,54,N,VIIRS,n,2.0NRT,268.25,3.85,D
2,58.21041,30.87205,305.40,0.52,0.41,2026-05-03,58,N,VIIRS,n,2.0NRT,277.90,0.97,N
3,58.21090,30.87533,304.22,0.52,0.41,2026-05-03,58,N,VIIRS,n,2.0NRT,278.08,1.02,N
4,58.70131,30.11929,317.81,0.48,0.40,2026-05-03,58,N,VIIRS,n,2.0NRT,276.30,2.85,N


In [40]:
##inspect and clean the data

# Check for missing values
fires_df.isnull().sum()

# Check confidence levels
fires_df['confidence'].value_counts()

confidence
n    115259
l     13801
h      5193
Name: count, dtype: int64

In [69]:
## convert coordinate into geometry
import geopandas as gpd

fires_gdf = gpd.GeoDataFrame(
    fires_df,
    geometry=gpd.points_from_xy(fires_df['longitude'], fires_df['latitude']),
    crs='EPSG:4326')

## load world and subset just africa
import os
os.getcwd() #prints current working directory
os.listdir("../data") # ... = go up one folder
os.listdir("../data/ne_110m_admin_0_countries")

world = gpd.read_file("../data/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp")
africa = world[world["CONTINENT"] == "Africa"]

fires_africa = gpd.sjoin(
    fires_gdf,
    africa[["geometry", "NAME"]],
    how="inner",
    predicate="within")


